# Importing librarys and data

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from datetime import datetime
from scipy.stats import spearmanr, kruskal, chi2_contingency
from itertools import combinations

In [ ]:
df = pd.read_csv("./src/data/Amazon Customer Behavior Survey.csv")

# Entendiendo el dataset

In [ ]:
df

In [ ]:
df.info()

El timestamp está como string y lo queremos en DataTime. Lo tenemos en cuenta para después.

In [ ]:
df.columns

In [ ]:
df.describe().T

Vemos que Personalized_Recomendation_Frequency, Parting_Accuracy y Shopping_Satisfaction tienes casi los mismos valores y comparten como mínimo y máximo el mismo número. Deducimos que seguramente sea de una encuesta en la que les hayan hecho elegir del 1 al 5. Podemos asignarles valores en la visualización. Lo tenemos en cuenta para después.

In [ ]:
df["Personalized_Recommendation_Frequency"].value_counts()

In [ ]:
df["Personalized_Recommendation_Frequency "].value_counts()

Vemos que hay dos columnas con el mismo nombre pero con distintos. La primera se trata de si alguna vez has hecho una compra basandote en recomendaciones personalizadas de Amazon y la segunda que tan a menudo recibes recomendaciones personalizadas de Amazon. Lo tenemos en cuenta para después.

In [ ]:
df.isnull().sum().sort_values(ascending=False)

In [ ]:
df["Product_Search_Method"].unique()

In [ ]:
df[df["Product_Search_Method"].isna()]

Vemos que en Product_Search_Method hay nulos. Lo tenemos en cuenta para después.

In [ ]:
df.age.describe()

In [ ]:
df["age"].value_counts().sort_index().head(6)

Vemos que hay menores haciendo compras. Lo tenemos en cuenta para después, expecialmente el de 3 años.

In [ ]:
df.Purchase_Frequency.unique()

In [ ]:
df.nunique()

In [ ]:
df["Purchase_Categories"].value_counts()

Nos preguntamos si podriamos encapsular algunas categorias o crear una nueva columna para una observación más exaustiva. Lo tenemos en cuenta para después.

##### Significado columnas

1. age = edad  
2. gender = género  
3. Purchase_Frequency = ¿Con qué frecuencia realizas compras en Amazon?  
4. Purchase_Categories = ¿Qué categorías de productos sueles comprar en Amazon?  
5. Personalized_Recommendation_Frequency = ¿Alguna vez has comprado un producto basándote en recomendaciones personalizadas de Amazon?  
6. Browsing_Frequency = ¿Con qué frecuencia navegas por el sitio web o la aplicación de Amazon?  
7. Product_Search_Method = ¿Cómo buscas productos en Amazon?  
8. Search_Result_Exploration = ¿Sueles explorar varias páginas de resultados de búsqueda o te quedas en la primera página?  
9. Customer_Reviews_Importance = ¿Qué importancia tienen las reseñas de clientes en tu proceso de toma de decisiones?  
10. Add_to_Cart_Browsing = ¿Añades productos a tu carrito mientras navegas por Amazon?  
11. Cart_Completion_Frequency = ¿Con qué frecuencia completas la compra después de añadir productos al carrito?  
12. Cart_Abandonment_Factors = ¿Qué factores influyen en tu decisión de abandonar una compra en el carrito?  
13. Saveforlater_Frequency = ¿Utilizas la función “Guardar para más tarde” de Amazon y, si es así, con qué frecuencia?  
14. Review_Left = ¿Alguna vez has dejado una reseña de un producto en Amazon?  
15. Review_Reliability = ¿Cuánto confías en las reseñas de productos al realizar una compra?  
16. Review_Helpfulness = ¿Encuentras información útil en las reseñas de otros clientes?  
17. Personalized_Recommendation_Frequency = ¿Con qué frecuencia recibes recomendaciones personalizadas de productos de Amazon?  
18. Recommendation_Helpfulness = ¿Te resultan útiles las recomendaciones?  
19. Rating_Accuracy = ¿Cómo calificarías la relevancia y precisión de las recomendaciones que recibes?  
20. Shopping_Satisfaction = ¿Qué tan satisfecho estás con tu experiencia general de compra en Amazon?  
21. Service_Appreciation = ¿Qué aspectos de los servicios de Amazon aprecias más?  
22. Improvement_Areas = ¿Hay áreas en las que crees que Amazon puede mejorar?  

In [ ]:
for col in df.columns:
    print(f"{col}:",df[col].unique())

Fijarnos en el Search_Result_Exploration nos podria dar indicios de compraciones de precio o duda a la hora de comprar. Lo tenemos en cuenta para después.

Vemos resultados repetidos o incoherentes en Service_Appreciation. Lo tenemos en cuenta para después.

Vemos resultados repetidos, con espacios a final o incoherentes en Improvement_Areas. Lo tenemos en cuenta para después.

In [ ]:
df.loc[df["Improvement_Areas"].isin(["UI", "Nil","Nothing"])]

Ya que son pocas columnas pensamos en cambiar los valores abstractos a "Nothing" ya que así manejaremos mejor los datos.

In [ ]:
df.describe(include="all").T


In [ ]:
df.duplicated().sum()

In [ ]:
df["Timestamp"].duplicated().sum()

In [ ]:
df["Timestamp"].value_counts(ascending=False)

In [ ]:
df[df["Timestamp"].isin(["2023/06/07 11:47:44 AM GMT+5:30"])]

Vemos que se hicieron a la misma hora pero son independientes. Descartamos la posibilidad de duplicados.

Finalmente creamos un diccionario para verlo mas fácil y rápido.

In [ ]:
# Genera un data dictionary automático

# Dividimos los tipos según el tipo de analisis que haremos a continuación
def var_type(col):
    if pd.api.types.is_numeric_dtype(df[col]):
        return "numérica"
    elif pd.api.types.is_datetime64_any_dtype(df[col]):
        return "fecha"
    else:
        return "categórica"

data_dict = pd.DataFrame({
    "columna": df.columns,
    "tipo": [var_type(c) for c in df.columns],
    "dtype": df.dtypes.astype(str).values,
    "nulos_pct": (df.isnull().sum()),
    "unicos": df.nunique().values,
    "min": [df[c].min() if pd.api.types.is_numeric_dtype(df[c]) else None for c in df.columns],
    "max": [df[c].max() if pd.api.types.is_numeric_dtype(df[c]) else None for c in df.columns],
    "duplicados": [df[c].duplicated().sum() for c in df.columns]})
#data_dict.to_csv("data_dictionary.csv", index=False)
data_dict

# Limpieza dataset

In [ ]:
# Convertir fechas en datetime
df["Timestamp"] = pd.to_datetime(df["Timestamp"], format="%Y/%m/%d %I:%M:%S %p GMT+5:30")

In [ ]:
# Limpiamos los nulos de Product_Search_Method transformandolos en la moda
df["Product_Search_Method"] = df["Product_Search_Method"].fillna(df["Product_Search_Method"].mode()[0])

In [ ]:
# Reasignamos otro nombre a la columna Personalized_Recommendation_Frequency 
df.rename(columns={"Personalized_Recommendation_Frequency": "Purchase_Based_on_Recommendations"}, inplace=True)

# Corregimos la columna Personalized_Recommendation_Frequency con espacio al final
df.rename(columns={"Personalized_Recommendation_Frequency ": "Personalized_Recommendation_Frequency"}, inplace=True)

In [ ]:
# Correguimos la columna Rating_Accuracy
df.rename(columns={"Rating_Accuracy ": "Rating_Accuracy"}, inplace=True)

In [ ]:
# Corregimos respuestas de Service_Appreciation quitando problemas de escritura y convirtiendo la . en la moda

df["Service_Appreciation"] = df["Service_Appreciation"].str.strip().str.capitalize().replace(".", df.Service_Appreciation.mode()[0])

In [ ]:
# Corregimos respuestas de Improvement_Areas quitando problemas de escritura y convirtiendo a "Nothing" los datos incoherentes

df["Improvement_Areas"] = df["Improvement_Areas"].str.strip().str.capitalize().replace([".","Ui", "Nil"], "Nothing")

In [ ]:
df.isnull().sum().sort_values(ascending=False)

In [ ]:
# Hacemos un binning en la edad/age

df["tramo_edad"] = pd.cut(df["age"],bins=[0, 17, 25, 35, 50, 65, 100],labels=["0-17","18-25", "26-35", "36-50", "51-65", "65+"])

In [ ]:
df["tramo_edad"].value_counts().sort_index()

In [ ]:
# Encodeamos ciertas columnas ya que mantienen un orden

ordinal_cols = {
    "Purchase_Frequency": ["Less than once a month", "Once a month", "Few times a month", "Once a week", "Multiple times a week"],
    "Browsing_Frequency": ["Rarely", "Few times a month", "Few times a week", "Multiple times a day"],
    "Cart_Completion_Frequency": ["Never", "Rarely", "Sometimes", "Often", "Always"],
    "Saveforlater_Frequency": ["Never", "Rarely", "Sometimes", "Often", "Always"],
    "Purchase_Based_on_Recommendations": ["No", "Sometimes", "Yes"],
    "Review_Reliability":["Never", "Rarely", "Occasionally", "Moderately", "Heavily"],
    "tramo_edad":["0-17", "18-25", "26-35", "36-50", "51-65", "65+"]
}

for col, order in ordinal_cols.items():
    df[col + "_enc"] = df[col].map({v: i+1 for i, v in enumerate(order)})

# Analysis

#### Analysis univariante

##### Númericas

In [ ]:
columnas_numericas = [col for col in df.columns if var_type(col) == "numérica"]

In [ ]:
def resumen_numerica(col):
    s = df[col]
    resumen = pd.DataFrame({
        "media":    [round(s.mean(), 3)],
        "mediana":  [round(s.median(), 3)],
        "moda":     [s.mode()[0]],
        "std":      [round(s.std(), 3)],
        "min":      [s.min()],
        "max":      [s.max()],
        "Q1":       [s.quantile(0.25)],
        "Q3":       [s.quantile(0.75)],
    }, index=[col])
    return resumen

pd.concat([resumen_numerica(c) for c in columnas_numericas])

In [ ]:
fig, axes = plt.subplots(len(columnas_numericas), 2, figsize=(14, 5 * len(columnas_numericas)))
colores = ["#7F77DD", "#1D9E75", "#378ADD", "#D85A30"]

for i, col in enumerate(columnas_numericas):
    s = df[col]
    color = colores[i % len(colores)]

    # Histograma
    ax1 = axes[i, 0]
    ax1.hist(s, bins=20, color=color, alpha=0.75, edgecolor="white", linewidth=0.5, density=True)
    s.plot.kde(ax=ax1, color=color, linewidth=2)
    ax1.axvline(s.mean(),   color="black",  linestyle="--", linewidth=1.2, label=f"Media: {s.mean():.2f}")
    ax1.axvline(s.median(), color="orange", linestyle=":",  linewidth=1.5, label=f"Mediana: {s.median():.2f}")
    ax1.set_title(f"{col} — Histograma")
    ax1.set_xlabel(col)
    ax1.set_ylabel("Densidad")
    ax1.legend(fontsize=9)

    # Texto tipo asimetría
    skew = s.skew()
    tipo_asim = "simétrica" if abs(skew) < 0.5 else ("sesgada derecha" if skew > 0 else "sesgada izquierda")
    ax1.text(0.97, 0.95, f"Asimetría: {skew:.2f}\n({tipo_asim})",
             transform=ax1.transAxes, ha="right", va="top", fontsize=8,
             bbox=dict(boxstyle="round,pad=0.3", facecolor="white", alpha=0.7))

    # Boxplot
    ax2 = axes[i, 1]
    bp = ax2.boxplot(s, vert=True, patch_artist=True, widths=0.5,
                     boxprops=dict(facecolor=color, alpha=0.6),
                     medianprops=dict(color="orange", linewidth=2),
                     whiskerprops=dict(color="gray"),
                     capprops=dict(color="gray"),
                     flierprops=dict(marker="o", color=color, alpha=0.5, markersize=5))
    ax2.set_title(f"{col} — Boxplot")
    ax2.set_ylabel(col)
    ax2.set_xticks([])

    # Anotar estadísticos
    q1, q3, med = s.quantile(0.25), s.quantile(0.75), s.median()
    iqr = q3 - q1
    outliers = s[(s < q1 - 1.5 * iqr) | (s > q3 + 1.5 * iqr)]
    ax2.text(0.97, 0.98,
             f"Q1: {q1:.1f}\nMediana: {med:.1f}\nQ3: {q3:.1f}\nIQR: {iqr:.1f}\nOutliers: {len(outliers)}",
             transform=ax2.transAxes, ha="right", va="top", fontsize=8,
             bbox=dict(boxstyle="round,pad=0.3", facecolor="white", alpha=0.7))

plt.suptitle("Análisis Univariante — Variables Numéricas", fontsize=16, fontweight="bold", y=1.01)
plt.tight_layout()
#plt.savefig("univariante_numericas.png", bbox_inches="tight", dpi=150)
plt.show()

**Conclusiones — Variables Numéricas:**

- **age**: Media de 30.8 años con mediana en 26, lo que indica una distribución sesgada a la derecha. El 75% de los encuestados tiene menos de 36 años, confirmando una audiencia joven. El rango es amplio (3–67 años) pero los extremos son casos anómalos.

- **Customer_Reviews_Importance**: Media de 2.48 sobre 5, con distribución bimodal (picos en 1 y 3). Esto sugiere dos perfiles: usuarios que no confían en reseñas y usuarios con confianza moderada. Pocos usuarios le dan máxima importancia (5).

- **Rating_Accuracy**: Media de 2.67, concentrada en valores centrales (2–3). La distribución es aproximadamente simétrica, lo que indica una percepción moderada y homogénea de la precisión de las valoraciones.

- **Shopping_Satisfaction**: Media de 2.46 sobre 5, la más baja de todas las métricas. La mediana es 2, lo que indica que más de la mitad de los usuarios tiene una satisfacción baja. Hay margen de mejora claro en la experiencia de compra.

In [ ]:
# Correlación de Spearman
corr_matrix = df[columnas_numericas].corr(method="spearman")

fig, ax = plt.subplots(figsize=(14, 10))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(
    corr_matrix,
    mask=mask,
    annot=True,
    fmt=".2f",
    cmap="RdYlGn",
    center=0,
    vmin=-1, vmax=1,
    linewidths=0.5,
    ax=ax,
    annot_kws={"size": 8}
)
ax.set_title("Correlación de Spearman — Variables Numéricas", fontweight="bold")
plt.tight_layout()
#plt.savefig("univariante_correlacion.png", bbox_inches="tight", dpi=150)
plt.show()

**Conclusiones — Correlación de Spearman:**

- **Rating_Accuracy ↔ Shopping_Satisfaction** (r=0.50, p<0.001): La correlación más fuerte del dataset. Los usuarios que perciben las valoraciones como más precisas reportan mayor satisfacción general. Esto sugiere que la fiabilidad del sistema de puntuaciones es clave para la experiencia de compra.

- **Shopping_Satisfaction ↔ Customer_Reviews_Importance** (r=0.42, p<0.001): Los usuarios más satisfechos tienden a dar más importancia a las reseñas, posiblemente porque las usan activamente y tienen buenas experiencias.

- **age ↔ Customer_Reviews_Importance** (r=0.08, p=0.059): Tendencia positiva muy débil y no significativa. No hay evidencia estadística de que los usuarios mayores dependan más de las reseñas.

- **age ↔ Shopping_Satisfaction** (r=-0.02, p=0.545): Sin relación. La satisfacción no varía con la edad.

##### Categóricas

In [ ]:
columnas_categoricas = [col for col in df.columns if var_type(col) == "categórica"]
columnas_categoricas

In [ ]:
cols_excluir = set(ordinal_cols.keys()) | {"Timestamp", "Purchase_Categories", "tramo_edad"}
nominales = [c for c in columnas_categoricas if c not in cols_excluir]
nominales

In [ ]:
# Coger los datos mas relevantes de Service_Appreciation e Improvement_Areas. 
# Tienen respuestas libres y hay muchas respuestas unicas.
top_cats = {
    "Service_Appreciation": [
        "Product recommendations", "Competitive prices",
        "Wide product selection", "User-friendly website/app interface"
    ],
    "Improvement_Areas": [
        "Customer service responsiveness", "Product quality and accuracy",
        "Reducing packaging waste", "Shipping speed and reliability"
    ]
}

In [ ]:
fig, axes = plt.subplots(6, 2, figsize=(16, 26))
axes = axes.flatten()
colores_nom = ["#7F77DD","#1D9E75","#378ADD","#D85A30","#D4537E",
               "#BA7517","#639922","#888780","#E24B4A","#4ABDD8"]

for i, col in enumerate(nominales):
    ax = axes[i]
    if col in top_cats:
        counts = df[col].value_counts()
        counts = counts[counts.index.isin(top_cats[col])]
    else:
        counts = df[col].value_counts()

    pct = counts / counts.sum() * 100
    bars = ax.bar(range(len(counts)), counts.values,
                  color=colores_nom[i % len(colores_nom)], alpha=0.8, edgecolor="white")

    for bar, val, p in zip(bars, counts.values, pct.values):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 1,
                f"{val}\n({p:.1f}%)", ha="center", va="bottom", fontsize=7.5)

    labels = [str(x)[:22] + "..." if len(str(x)) > 22 else str(x) for x in counts.index]
    ax.set_xticks(range(len(counts)))
    ax.set_xticklabels(labels, rotation=20, ha="right", fontsize=8)
    ax.set_title(col.replace("_", " "), fontweight="bold")
    ax.set_ylabel("Frecuencia")
    ax.set_ylim(0, counts.max() * 1.25)

plt.suptitle("Variables Categóricas Nominales", fontsize=15, fontweight="bold", y=1.01)
plt.tight_layout()
#plt.savefig("univariante_nominales.png", bbox_inches="tight", dpi=150)
plt.show()

**Conclusiones — Variables Categóricas Nominales:**

- **Gender**: Muestra predominantemente femenina (58.5%). El 14.8% prefiere no indicar su género, lo que puede sesgar los análisis por género.

- **Cart_Abandonment_Factors**: El 42.4% abandona por encontrar mejor precio en otro sitio y el 40% por cambio de opinión. El precio competitivo es el principal reto para retener al comprador, más que el coste de envío (11.6%).

- **Review_Left**: La comunidad está dividida casi al 50% — el 51.5% deja reseñas y el 48.5% no. Esto indica una base activa pero con mucho potencial de activación.

- **Review_Helpfulness**: Solo el 39.4% considera las reseñas siempre útiles. El 22.9% no las encuentra útiles, lo que sugiere un problema de calidad o relevancia de las reseñas.

- **Recommendation_Helpfulness**: Las recomendaciones personalizadas generan escepticismo — solo el 26.1% las considera útiles y el 28.6% directamente no las encuentra útiles.

- **Service_Appreciation**: Los usuarios valoran principalmente las recomendaciones de producto (30.7%) y los precios competitivos (30.2%), ambos casi empatados.

- **Improvement_Areas**: La atención al cliente es la principal área de mejora demandada (36.1%), seguida de la calidad del producto (26.4%).

In [ ]:
colores_ord = ["#7F77DD", "#1D9E75", "#378ADD", "#D85A30", "#BA7517", "#D4537E"]

for i, col in enumerate(ordinal_cols.keys()):
    fig, ax = plt.subplots(figsize=(10, 3.5))
    orden = ordinal_cols[col]
    counts = df[col].value_counts().reindex(orden, fill_value=0)
    pct = counts / counts.sum() * 100

    bars = ax.barh(orden, counts.values, color=colores_ord[i % len(colores_ord)],
                   alpha=0.8, edgecolor="white", height=0.6)

    for bar, val, p in zip(bars, counts.values, pct.values):
        ax.text(bar.get_width() + 2, bar.get_y() + bar.get_height() / 2,
                f"{val} ({p:.1f}%)", va="center", fontsize=9)

    ax.set_title(col.replace("_", " "), fontweight="bold")
    ax.set_xlim(0, max(counts.max() * 1.3, 10))

    plt.tight_layout()
    #plt.savefig(f"univariante_ordinales_{col}.png", bbox_inches="tight", dpi=150)
    plt.show()

**Conclusiones — Variables Ordinales:**

- **Purchase_Frequency**: El 33.7% compra pocas veces al mes — el segmento más grande. Solo el 9.3% compra varias veces a la semana, lo que indica que los compradores de alta frecuencia son una minoría valiosa.

- **Browsing_Frequency**: El 41.4% navega pocas veces a la semana. Hay más usuarios que navegan con frecuencia de la que compran, lo que confirma la existencia de un segmento de navegadores no compradores.

- **Cart_Completion_Frequency**: El 50.5% completa la compra solo a veces. Solo el 7.8% siempre completa la compra, lo que indica altas tasas de abandono de carrito como patrón estructural.

- **Saveforlater_Frequency**: El 41.7% guarda para más tarde a veces. Es una función moderadamente usada, lo que sugiere indecisión o comparación de precios.

- **Review_Reliability**: El 33.1% confía moderadamente en las reseñas y el 31.6% solo ocasionalmente. Solo el 24.8% confía mucho en ellas, lo que refleja escepticismo generalizado.

- **tramo_edad**: El 43.5% tiene entre 18 y 25 años, lo que convierte a los jóvenes adultos en el segmento dominante. Los mayores de 50 años representan menos del 5% de la muestra.

In [ ]:
cat_exploded = df["Purchase_Categories"].str.split(";").explode().str.strip()
cat_counts = cat_exploded.value_counts()

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.barh(cat_counts.index, cat_counts.values,
               color="#7F77DD", alpha=0.8, edgecolor="white", height=0.6)

for bar, val in zip(bars, cat_counts.values):
    ax.text(bar.get_width() + 2, bar.get_y() + bar.get_height() / 2,
            f"{val} ({val/602*100:.1f}% usuarios)", va="center", fontsize=9)

ax.set_title("Purchase Categories — Menciones por categoría", fontweight="bold")
ax.set_xlabel("Nº de menciones")
ax.set_xlim(0, cat_counts.max() * 1.4)
ax.invert_yaxis()
plt.tight_layout()
#plt.savefig("univariante_categories.png", bbox_inches="tight", dpi=150)
plt.show()

In [ ]:
# Nº medio de categorías por usuario
n_cats = df["Purchase_Categories"].str.split(";").apply(len)
print(f"Media de categorías por usuario: {n_cats.mean():.2f}")
print(f"Máximo: {n_cats.max()} | Mínimo: {n_cats.min()}")
print(cat_counts.sort_index())

**Conclusiones — Purchase Categories:**

- **Clothing and Fashion** (29.6%) y **Beauty and Personal Care** (27.5%) son las categorías más compradas, reflejando el perfil mayoritariamente femenino y joven de la muestra.

- **Home and Kitchen** (19%) ocupa el tercer puesto, lo que indica un segmento de compradores del hogar relevante.

- **Groceries and Gourmet Food** solo aparece en el 9.6% de las menciones, lo que sugiere que Amazon no es el canal principal para alimentación en esta muestra.

- Los usuarios compran de media en más de una categoría, lo que indica comportamiento de compra diversificado.

#### Analysis bivariante

In [ ]:
def significancia(p):
    if p < 0.001: return "***"
    if p < 0.01:  return "**"
    if p < 0.05:  return "*"
    return "ns"

##### *Numérica vs Numérica*

In [ ]:
cols_num_plot = ["age", "Customer_Reviews_Importance", "Rating_Accuracy", "Shopping_Satisfaction"]
pares = list(combinations(cols_num_plot, 2))

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 9))
axes = axes.flatten()

for i, (x, y) in enumerate(pares):
    ax = axes[i]
    r, p = spearmanr(df[x], df[y])
    sns.regplot(data=df, x=x, y=y, ax=ax,
                scatter_kws={"alpha":0.2,"s":15}, line_kws={"color":"red","lw":1.5})
    ax.set_title(f"{x.replace("_"," ")} vs {y.replace("_"," ")}", fontsize=10, fontweight="bold")
    ax.set_xlabel(x.replace("_"," "), fontsize=9)
    ax.set_ylabel(y.replace("_"," "), fontsize=9)
    ax.annotate(f"r={r:.3f} {significancia(p)}\np={p:.4f}", xy=(0.97,0.97),
                xycoords="axes fraction", ha="right", va="top", fontsize=9,
                bbox=dict(boxstyle="round", facecolor="white", alpha=0.8))

for j in range(len(pares), len(axes)):
    axes[j].set_visible(False)

plt.suptitle("Numérica vs Numérica", fontsize=13, fontweight="bold")
plt.tight_layout()
#plt.savefig("biv_num_num.png", bbox_inches="tight", dpi=150)
plt.show()

**Conclusiones — Scatter Numérica vs Numérica:**

- **Rating_Accuracy ↔ Shopping_Satisfaction** (r=0.50, ***): La relación más fuerte. Cuando los usuarios perciben las valoraciones como más precisas, su satisfacción aumenta significativamente. Mejorar el sistema de ratings tendría impacto directo en la satisfacción.

- **Shopping_Satisfaction ↔ Customer_Reviews_Importance** (r=0.42, ***): Los usuarios satisfechos valoran más las reseñas, lo que puede indicar que las reseñas contribuyen a tomar mejores decisiones de compra.

- **Rating_Accuracy ↔ Customer_Reviews_Importance** (r=0.30, ***): Correlación moderada. Quien más usa reseñas también presta más atención a la precisión de las valoraciones — son perfiles coherentes.

- **age**: No muestra correlación significativa con ninguna de las otras variables numéricas, lo que sugiere que la edad no determina por sí sola la experiencia o el comportamiento en Amazon.

##### *Numérica vs Categórica — Boxplots*

In [ ]:
pares_nc = [
    ("Shopping_Satisfaction", "Gender"),
    ("Shopping_Satisfaction", "Purchase_Frequency"),
    ("Shopping_Satisfaction", "Browsing_Frequency"),
    ("Shopping_Satisfaction", "Cart_Abandonment_Factors"),
    ("age",                   "Purchase_Frequency"),
    ("age",                   "Browsing_Frequency"),
    ("age",                   "tramo_edad"),
    ("Customer_Reviews_Importance", "Review_Left"),
    ("Customer_Reviews_Importance", "Review_Reliability"),
    ("Customer_Reviews_Importance", "Review_Helpfulness"),
    ("Rating_Accuracy",       "Review_Helpfulness"),
    ("Rating_Accuracy",       "Review_Reliability"),
]


for num, cat in pares_nc:
    orden = ordinal_cols.get(cat, sorted(df[cat].dropna().unique()))
    grupos = [df[df[cat]==g][num].dropna() for g in orden if len(df[df[cat]==g][num].dropna()) > 0]
    labels = [o for o in orden if len(df[df[cat]==o][num].dropna()) > 0]

    stat, p = kruskal(*grupos) if len(grupos) >= 2 else (0, 1)

    fig, ax = plt.subplots(figsize=(10, 4))
    sns.boxplot(data=df, x=cat, y=num, order=labels, palette="Set2", ax=ax,
            hue=cat, legend=False, flierprops=dict(marker="o", markersize=3, alpha=0.4))
    ax.set_title(f"{num.replace("_"," ")} por {cat.replace("_"," ")}\nH={stat:.2f}, p={p:.4f} {significancia(p)}",
                 fontweight="bold", fontsize=11)
    ax.set_xlabel(cat.replace("_"," "))
    ax.set_ylabel(num.replace("_"," "))
    ax.set_xticks(range(len(labels)))
    ax.set_xticklabels([l[:20] for l in labels], rotation=15, ha="right", fontsize=8)
    plt.tight_layout()
    #plt.savefig(f"biv_{num}_vs_{cat}.png", bbox_inches="tight", dpi=150)
    plt.show()

**Conclusiones — Numérica vs Categórica:**

- **Satisfacción por Género** (H=8.72, p=0.033, *): Diferencias significativas. El grupo 'Others' tiene la mayor satisfacción media (2.84) y los hombres la menor (2.30). Las mujeres se sitúan en la media general (2.46).

- **Satisfacción por Purchase_Frequency** (H=2.67, p=0.615, ns): Sin diferencias significativas. La frecuencia de compra no determina la satisfacción — compradores habituales y esporádicos reportan niveles similares.

- **Satisfacción por Browsing_Frequency** (H=12.98, p=0.005, **): Diferencias significativas. Los usuarios que navegan raramente tienen mayor satisfacción media (2.71) que los que navegan varias veces al día (2.12). Navegar mucho sin comprar puede generar frustración.

- **Edad por Purchase_Frequency** (H=8.36, p=0.079, ns): Tendencia leve sin significación estadística. No hay evidencia clara de que la edad determine la frecuencia de compra.

- **Customer_Reviews_Importance por Review_Left** (H=0.09, p=0.761, ns): Sin diferencias. La importancia que se da a las reseñas no predice si el usuario las deja o no.

- **Customer_Reviews_Importance por Review_Reliability** (H=115.30, p<0.001, ***): Relación muy significativa e inversa. Los que confían 'Heavily' en reseñas les dan paradójicamente menos importancia en su decisión (media 1.76), mientras que los que 'Never' confían les dan más importancia (media 3.65). Esto sugiere que confiar en reseñas y darles importancia son conceptos distintos.

##### *Categórica vs Categórica — Barras apiladas + Chi²*

In [ ]:
pares_cc = [
    ("Gender",              "Purchase_Frequency"),
    ("Gender",              "Browsing_Frequency"),
    ("Gender",              "Review_Left"),
    ("Browsing_Frequency",  "Cart_Completion_Frequency"),
    ("Browsing_Frequency",  "Purchase_Frequency"),
    ("Purchase_Frequency",  "Cart_Completion_Frequency"),
    ("Purchase_Frequency",  "Cart_Abandonment_Factors"),
    ("Review_Left",         "Review_Reliability"),
    ("Review_Left",         "Cart_Abandonment_Factors"),
    ("Review_Helpfulness",  "Recommendation_Helpfulness"),
    ("Add_to_Cart_Browsing","Cart_Completion_Frequency"),
]

for cat1, cat2 in pares_cc:
    ct = pd.crosstab(df[cat1], df[cat2])
    ct_pct = ct.div(ct.sum(axis=1), axis=0) * 100
    chi2, p, _, _ = chi2_contingency(ct)
    v = np.sqrt(chi2 / (ct.values.sum() * (min(ct.shape) - 1)))

    fig, ax = plt.subplots(figsize=(10, 4.5))
    ct_pct.plot(kind="bar", stacked=True, ax=ax, colormap="Set2", edgecolor="white", width=0.6)
    ax.set_title(f"{cat1.replace("_"," ")} vs {cat2.replace("_"," ")}\nChi²={chi2:.2f}, p={p:.4f} {significancia(p)} | Cramér\"s V={v:.3f}",
                 fontweight="bold", fontsize=11)
    ax.set_xlabel(cat1.replace("_"," "))
    ax.set_ylabel("% dentro del grupo")
    ax.set_xticklabels(ax.get_xticklabels(), rotation=15, ha="right", fontsize=8)
    ax.legend(title=cat2.replace("_"," "), bbox_to_anchor=(1.01, 1), loc="upper left", fontsize=8)
    ax.set_ylim(0, 115)
    plt.tight_layout()
    #plt.savefig(f"biv_{cat1}_vs_{cat2}.png", bbox_inches="tight", dpi=150)
    plt.show()

**Conclusiones — Categórica vs Categórica:**

- **Browsing_Frequency ↔ Purchase_Frequency** (Chi²=134.16, p<0.001, V=0.273): Asociación significativa y moderada. Los usuarios que navegan más frecuentemente también compran con mayor frecuencia, aunque la relación no es perfecta — existe un segmento que navega sin comprar.

- **Browsing_Frequency ↔ Cart_Completion_Frequency** (Chi²=80.62, p<0.001, V=0.211): Asociación significativa. Los que navegan varias veces al día tienen una tasa de completitud de carrito notablemente superior a los que navegan raramente. La navegación frecuente está asociada a mayor intención de compra.

- **Add_to_Cart_Browsing ↔ Cart_Completion_Frequency** (Chi²=40.63, p<0.001, V=0.184): Los usuarios que añaden productos al carrito mientras navegan tienden a completar más compras. Añadir al carrito es un indicador de intención de compra real.

- **Review_Left ↔ Review_Reliability** (Chi²=17.32, p=0.002, V=0.170): Los usuarios que dejan reseñas confían más en ellas. Hay coherencia entre el comportamiento de escribir reseñas y la confianza en el sistema.

- **Review_Helpfulness ↔ Recommendation_Helpfulness** (asociación significativa): Los usuarios que encuentran útiles las reseñas también tienden a encontrar útiles las recomendaciones personalizadas — perfil de usuario receptivo a la información social.

##### *Resumen estadístico de todos los pares*

In [ ]:
resultados = []

for x, y in combinations(columnas_numericas, 2):
    r, p = spearmanr(df[x], df[y])
    resultados.append({"var1":x,"var2":y,"test":"Spearman","estadístico":round(r,4),"p_valor":round(p,4),"significancia":significancia(p)})

for num, cat in pares_nc:
    grupos = [df[df[cat]==g][num].dropna() for g in df[cat].dropna().unique() if len(df[df[cat]==g][num].dropna()) > 0]
    if len(grupos) >= 2:
        stat, p = kruskal(*grupos)
        resultados.append({"var1":num,"var2":cat,"test":"Kruskal-Wallis","estadístico":round(stat,4),"p_valor":round(p,4),"significancia":significancia(p)})

for cat1, cat2 in pares_cc:
    ct = pd.crosstab(df[cat1], df[cat2])
    chi2, p, _, _ = chi2_contingency(ct)
    v = np.sqrt(chi2 / (ct.values.sum() * (min(ct.shape) - 1)))
    resultados.append({"var1":cat1,"var2":cat2,"test":"Chi²","estadístico":round(chi2,4),"p_valor":round(p,4),"significancia":significancia(p)})

df_res = pd.DataFrame(resultados).fillna("-").sort_values("p_valor")
#df_res.to_csv("bivariante_resultados.csv", index=False)
df_res

**Conclusiones — Resumen estadístico:**

- Las relaciones más fuertes y significativas del dataset son **Browsing_Frequency ↔ Purchase_Frequency** (V=0.273) y **Rating_Accuracy ↔ Shopping_Satisfaction** (r=0.50), ambas con p<0.001.

- La mayoría de los pares cat-cat muestran asociaciones significativas pero con Cramér's V por debajo de 0.30, lo que indica asociaciones estadísticamente reales pero de tamaño de efecto pequeño-moderado.

- Las variables de edad no muestran relaciones significativas con casi ninguna otra variable, lo que pone en duda las hipótesis basadas en diferencias generacionales.

#### Analysis multivariante

# Hipotesis